<a href="https://colab.research.google.com/github/SampMark/Deep-Learning/blob/main/Hyperparameter_Tuning_using_Advanced_Training_Loops_with_KerasTuner_and_Hyperband.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Ajuste de Hiperparâmetros com a implementação de loops de treinamento avançados, usando o KerasTuner e o HyperbandOracle**

Neste notebook iremos demonstrar e explorar uma poderosa combinação de **loop de treinamento personalizado no** TensorFlow/Keras com as capacidades de otimização de hiperparâmetros (HPO) do `KerasTuner`, utilizando o `HyperbandOracle` para guiar a busca. O script é estruturado para realizar a otimização de hiperparâmetros de um modelo de classificação para o _dataset_ MNIST, utilizando um loop de treinamento totalmente customizado e integrado ao KerasTuner.

### **Uma Introdução sobre a importãncia da flexibilidade e eficiência no treinamento de modelos**

No desenvolvimento de modelos de aprendizado de máquina, especialmente com redes neurais profundas, duas facetas são cruciais: a flexibilidade no processo de treinamento e a eficiência na busca pelos melhores hiperparâmetros. Loops de treinamento personalizados oferecem um controle granular, permitindo a implementação de lógicas de treinamento complexas, como em Redes Adversariais Generativas (GANs) ou aprendizado por reforço, que vão além do que o método model.fit() padrão pode oferecer.

Por outro lado, a otimização de hiperparâmetros (HPO) é essencial, pois o desempenho de um modelo é altamente sensível as escolhas efetuadas, tais como a taxa de aprendizado, o tamanho do lote (_batch size_) e arquitetura da rede. Equívocos na escolha de hiperparâmentros elevam o custo computacional associado ao treinamento de múltiplos modelos durante etapa de oti,ização, o pode ser algo proibitivo. É aqui que algoritmos eficientes como o **Hyperband** (Li et al., 2017, 2018) entram em cena.

O Hyperband, fundamentado na teoria dos _multi-armed bandits_, emprega uma estratégia de alocação adaptativa de recursos e parada antecipada (_early-stopping_) para descartar rapidamente configurações de hiperparâmetros pouco promissoras, focando os recursos computacionais naquelas com maior potencial.

### **Loops de Treinamento Personalizados com KerasTuner e Hyperband**

O código a seguir implementa um loop de treinamento do zero, controlando explicitamente o _forward pass_, o cálculo da perda, a retropropagação dos gradientes (`tf.GradientTape`) e atualização dos pesos do modelo (`optimizer.apply_gradients`). Esta configuração oferece máxima flexibilidade, conforme discutido, sendo essencial para cenários além do model.fit() padrão.

O `KerasTuner` orquestra a busca por hiperparâmetros e a classe `CustomLoopTuner` herda de `kt.Tuner` para permitir que essa orquestração ocorra sobre o loop de treinamento personalizado.

#### **O Hyperband**

O `HyperbandOracle` implementa o algoritmo Hyperband, sugerido por LI _et al_. (2017 e 2018). Sua lógica principal é a alocação adaptativa de recursos. Em vez de treinar todas as configurações de hiperparâmetros por um número fixo e longo de épocas, o Hyperband começa com muitas configurações e as treina por poucas épocas (reduzindo a necessidade de recursos).
O Hyperband então descarta uma fração das configurações de pior desempenho (definida pelo factor) e aloca mais recursos (mais épocas) às sobreviventes. Esse processo é chamado de `SuccessiveHalving`.

O Hyperband executa múltiplos _brackets_ de `SuccessiveHalving`, cada um começando com um número diferente de configurações e um "orçamento inicial" de recursos diferente. O que p torna robusto à escolha inicial de "agressividade" do _early-stopping_.

No código a seguir, o `HyperbandOracle` decide quantas `epochs` cada trial deve rodar (`run_trial`) . O `run_trial` simplesmente executa o treinamento por esse número de épocas e reporta a métrica de validação.

**A eficiência do Hyperband vem de não desperdiçar tempo com configurações ruins**, permitindo uma exploração mais ampla do espaço de hiperparâmetros dentro de um orçamento computacional fixo.

In [1]:
# Aloque uma GPU
!nvidia-smi

Thu May 15 21:11:43 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             47W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
!pip install tensorflow

In [3]:
!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 3.9 MB/s eta 0:00:00


In [4]:
# -*- coding: utf-8 -*-
# --- Bibliotecas necessárias ---
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.metrics import SparseCategoricalAccuracy, Mean
import keras_tuner as kt
from sklearn.model_selection import train_test_split

## **1. Configurações Globais e Preparação de Dados**

### **Configurações Globais**

* `RANDOM_SEED`: garante a reprodutibilidade dos resultados.
* `BATCH_SIZE`: tamanho do lote padrão para os datasets, o valor pode ser ajustado pelo KerasTuner se definido como um hiperparâmetro.
* `NUM_EPOCHS`: define o número máximo de épocas que o Hyperband pode alocar para uma única configuração no seu bracket mais longo. Não é o número total de épocas da busca.
* `LEARNING_RATE`: taxa de aprendizado padrão, que também será otimizada.

### **Preparação de Dados**

1. O dataset MNIST é carregado e os pixels das imagens são normalizados para o intervalo [0, 1].
2. O conjunto de treinamento original é dividido em subconjuntos de treinamento e validação.
3. Os dados são convertidos em objetos `tf.data.Dataset`, algo crucial para a eficiência, pois permite o uso de `shuffle()` para embaralhar os dados de treino, `batch()` para agrupar em lotes, e `prefetch(tf.data.AUTOTUNE)` para pré-buscar dados e otimizar o pipeline de entrada, mantendo a GPU alimentada.

In [5]:
# --- Configurações Globais ---
RANDOM_SEED = 42
BATCH_SIZE = 64 # Batch size inicial para os datasets, pode ser sobrescrito pelo tuner
NUM_EPOCHS = 10 # Número de épocas para o Hyperband considerar como max_epochs
LEARNING_RATE = 1e-3 # Taxa de aprendizado default, será ajustada pelo tuner

# Semente para reprodutibilidade
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# --- Preparação de Dados (MNIST como exemplo para o loop customizado) ---
(x_train_full, y_train_full), (x_test, y_test) = mnist.load_data()

# Normalização
x_train_full = x_train_full.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Divisão do conjunto de treinamento completo em treino e validação
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=0.1, random_state=RANDOM_SEED
)

# Criação de tf.data.Dataset
# O batch_size aqui é um default, o tuner poderá usar um HP para isso.
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(buffer_size=len(x_train)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_dataset = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


## **2. Classe CustomLoopTuner**

Esta é a peça central da integração, pois herda de `kt.Tuner` a classe base do KerasTuner para criar estratégias de busca personalizadas.

O método mais importante sobrescrito é o `run_trial`, chamado pelo KerasTuner para cada combinação de hiperparâmetros (um "trial") que ele decide testar.

### **Método `run_trial`**
1. `hp = trial.hyperparameters`: acessa o objeto `HyperParameters` para o trial atual, que contém os valores dos hiperparâmetros selecionados pelo `HyperbandOracle` para esta execução específica.

2. `current_batch_size`: um hiperparâmetro para o tamanho do lote é definido (ou recuperado se já definido na função `build_model`). Os datasets de treino e validação são então recriados (`unbatch().batch()`) com este `batch_size` específico do trial.

3. `model = self.hypermodel.build(hp)`: constrói a arquitetura do modelo, `self.hypermodel` refere-se à função passada para o construtor do `CustomLoopTuner` (neste caso, `build_model_for_custom_loop`), que define como o modelo é construído com base nos hiperparâmetros.

4. **Otimizador e Função de Perda**: e otimizador `adam` é instanciado com a `learning_rate` obtida dos hiperparâmetros do trial. A função de perda `SparseCategoricalCrossentropy` é escolhida, adequada para classificação multiclasse com rótulos inteiros e saídas de logits.

5. **Métricas** para acompanhar a perda e acurácia durante o treino e validação são instanciadas, `Mean` para perdas e `SparseCategoricalAccuracy` para acurácias.

### **Funções `train_step_custom` e `val_step_custom`**

Estas funções implementam a lógica para um único passo de treinamento e um único passo de validação, respectivamente.

* `@tf.function`: compila a função Python em um grafo TensorFlow, o que geralmente resulta em uma execução significativamente mais rápida, especialmente para operações repetitivas como os passos de treinamento.

* `train_step_custom`:
  * Utiliza `tf.GradientTape()` para registrar as operações durante o _forward pass_ (`logits = model(x, training=True)`). O `training=True` é importante para camadas como `BatchNormalization` ou `Dropout` se comportarem corretamente durante o treino.
  * Calcula a perda.
  * Adiciona quaisquer perdas de regularização que o modelo possa ter (definidas através de `kernel_regularizer`, `bias_regularizer` ou `activity_regularizer` nas camadas, ou via `model.add_loss()`).
  * Calcula os gradientes da perda em relação aos pesos treináveis do modelo (`tape.gradient(...)`).
  * Aplica os gradientes para atualizar os pesos (`optimizer.apply_gradients(...)`).
  * Atualiza as métricas de treinamento
  
* `val_step_custom`:
  * Realiza o `_forward pass_ com `training=False` para garantir que camadas como `Dropout` estejam desativadas.
  * Calcula a perda e atualiza as métricas de validação.
  
### **Loop de Épocas e Reporte ao Oracle**

1. O loop externo itera pelo número de `epochs` que o `HyperbandOracle` determinou para este trial específico naquele estágio da busca.
2. No início de cada época, as métricas são resetadas.
3. Loops internos iteram sobre os _datasets_ de treino e validação, chamando `train_step_custom` e `val_step_custom` respectivamente.
4. Em seguida ocorre o **reporte de métricas intermediárias**:
```
self.oracle.update_trial(trial.trial_id, metrics={self.oracle.objective.name: float(current_val_accuracy_epoch)}, step=epoch)
```

É chamado ao final de cada época, permitindo que o Oracle (e, consequentemente, o algoritmo Hyperband) tenham acesso ao desempenho da configuração ao longo do tempo. O Hyperband usa essa informação para decidir quais configurações descartar prematuramente. O `step=epoch` informa ao `KerasTuner` a qual época a métrica se refere.

5. Por fim, é reportada a métrica final após todas as épocas do trial serem concluídas.
```
self.oracle.update_trial(trial.trial_id, {self.oracle.objective.name: float(final_val_accuracy)})
```
É chamado novamente, mas desta vez sem o argumento `step`, sinalizando o valor final da métrica objetivo para este trial. O nome da métrica (`self.oracle.objective.name`) é acessado diretamente do Oracle para garantir consistência.

In [6]:
# #######################################################################################
# @title **Loops de Treinamento Personalizados com KerasTuner e Hyperband - Avançado**
# #######################################################################################

class CustomLoopTuner(kt.Tuner):
    def run_trial(self, trial, train_dataset, val_dataset, epochs, **fit_kwargs):
        """
        Executa um trial de treinamento com um loop customizado.

        Args:
            trial: Um objeto Trial do KerasTuner.
            train_dataset: tf.data.Dataset para treinamento.
            val_dataset: tf.data.Dataset para validação.
            epochs: Número de épocas para treinar (determinado pelo Oracle).
            **fit_kwargs: Argumentos adicionais (não usados diretamente aqui, mas parte da assinatura).
        """
        hp = trial.hyperparameters

        current_batch_size = hp.Int("batch_size", min_value=32, max_value=128, step=32, default=BATCH_SIZE)

        current_train_dataset = train_dataset.unbatch().batch(current_batch_size).prefetch(tf.data.AUTOTUNE)
        current_val_dataset = val_dataset.unbatch().batch(current_batch_size).prefetch(tf.data.AUTOTUNE)

        model = self.hypermodel.build(hp)

        learning_rate = hp.get('learning_rate')
        optimizer = Adam(learning_rate=learning_rate)
        loss_fn_custom = SparseCategoricalCrossentropy(from_logits=True)

        train_loss_metric_custom = Mean(name='train_loss')
        train_accuracy_metric_custom = SparseCategoricalAccuracy(name='train_accuracy')
        val_loss_metric_custom = Mean(name='val_loss')
        val_accuracy_metric_custom = SparseCategoricalAccuracy(name='val_accuracy')

        @tf.function
        def train_step_custom(x, y):
            with tf.GradientTape() as tape:
                logits = model(x, training=True)
                loss = loss_fn_custom(y, logits)
                if model.losses:
                    loss += tf.add_n(model.losses)
            grads = tape.gradient(loss, model.trainable_weights)
            optimizer.apply_gradients(zip(grads, model.trainable_weights))
            train_loss_metric_custom.update_state(loss)
            train_accuracy_metric_custom.update_state(y, logits)

        @tf.function
        def val_step_custom(x, y):
            logits = model(x, training=False)
            loss = loss_fn_custom(y, logits)
            if model.losses:
                loss += tf.add_n(model.losses)
            val_loss_metric_custom.update_state(loss)
            val_accuracy_metric_custom.update_state(y, logits)

        # Loop de épocas e reporte ao Oracle
        for epoch in range(epochs):
            print(f"    Trial {trial.trial_id} - Época {epoch + 1}/{epochs}")
            train_loss_metric_custom.reset_state()
            train_accuracy_metric_custom.reset_state()
            val_loss_metric_custom.reset_state()
            val_accuracy_metric_custom.reset_state()

            for step_train, (x_batch, y_batch) in enumerate(current_train_dataset):
                train_step_custom(x_batch, y_batch)
                if step_train % 200 == 0:
                     print(f"      Batch {step_train}: Perda Treino (acumulada na época) = {train_loss_metric_custom.result():.4f}, Acurácia Treino (acumulada na época) = {train_accuracy_metric_custom.result():.4f}")

            for x_val_batch, y_val_batch in current_val_dataset:
                val_step_custom(x_val_batch, y_val_batch)

            current_val_accuracy_epoch = val_accuracy_metric_custom.result().numpy()
            print(f"    Resultado da Época {epoch + 1} (Trial {trial.trial_id}):")
            print(f"      Perda Treino: {train_loss_metric_custom.result():.4f}, Acurácia Treino: {train_accuracy_metric_custom.result():.4f}")
            print(f"      Perda Validação: {val_loss_metric_custom.result():.4f}, Acurácia Validação: {current_val_accuracy_epoch:.4f}")

            # Reportar métricas intermediárias ao Oracle
            self.oracle.update_trial(trial.trial_id, metrics={self.oracle.objective.name: float(current_val_accuracy_epoch)}, step=epoch)

        final_val_accuracy = val_accuracy_metric_custom.result().numpy()

        # Reporte da métrica final, após todas as épocas do trial serem concluídas
        self.oracle.update_trial(trial.trial_id, {self.oracle.objective.name: float(final_val_accuracy)})

## **3. Função `build_model_for_custom_loop`**

Esta função é o hypermodel que o KerasTuner utiliza.1. Primeiramente ela recebe um objeto `hp` (`keras_tuner.HyperParameters`) como argumento. 2. Dentro dela, foram definidos os hiperparâmetros que se deseja ajustar usando os métodos de `hp`, como:
  * `hp.Int('units_custom', ...)`: define um hiperparâmetro inteiro para o número de unidades na primeira camada densa, variando de 64 a 256 em passos de 64.
  * `hp.Float('learning_rate', ...)`: define um hiperparâmetro de ponto flutuante para a taxa de aprendizado, amostrado de uma distribuição logarítmica entre `1e-4` e `1e-2`.
* A função constrói um modelo Keras Sequencial simples para classificação no MNIST.
O modelo retornado por esta função não é compilado.  A compilação com a definição do otimizador, função de perda e métricas ocorre dentro do método `run_trial` da classe `CustomLoopTuner`, pois o otimizador (e sua taxa de aprendizado) é um dos hiperparâmetros.


### **Instanciação do `CustomLoopTuner`**:

1. `hypermodel`: a função `build_model_for_custom_loop` é passada.
2. `oracle`: uma instância de `kt.oracles.HyperbandOracle` é usada.
  * `objective=kt.Objective('val_accuracy', direction='max')`: especifica que o objetivo da otimização é maximizar a métrica `val_accuracy`, a quel deve corresponder exatamente ao nome da métrica reportada em `self.oracle.update_trial`.
  * `max_epochs=NUM_EPOCHS`: informa ao Hyperband o orçamento máximo de épocas para a configuração mais "cara" (no _bracket_ que treina por mais tempo).
  * `factor=3`: é o parâmetro do Hyperband que controla a taxa de descarte, a cada rodada do `SuccessiveHalving` (dentro de um _bracket_ do Hyperband), `1/factor` das configurações são mantidas e o número de épocas é multiplicado por `factor`.
  * `seed` para reprodutibilidade da busca.
* `directory` e `project_name`: serve para armazenar os resultados da busca.
  * `overwrite=True`: sobrescreve resultados de buscas anteriores com o mesmo nome de projeto e diretório.
* `tuner_custom.search_space_summary()`: imprime um resumo dos hiperparâmetros que serão ajustados e seus respectivos intervalos/valores.
* `tuner_custom.search(...)`: inicia o processo de otimização.
  * `train_dataset` e `val_dataset` são os datasets de treinamento e validação.
  * `epochs=NUM_EPOCHS`: quando passado para o método search de um tuner que usa Hyperband, informa ao Hyperband qual é o orçamento máximo de épocas por configuração. O Hyperband então gerencia internamente quantas épocas cada trial específico deve rodar em cada estágio ("rung") de seus _brackets_.

In [7]:
def build_model_for_custom_loop(hp):
    model = Sequential([
        Input(shape=(28, 28), name="input_layer"),
        Flatten(name="flatten_layer"),
        Dense(hp.Int('units_custom', min_value=64, max_value=256, step=64, default=128),
              activation="relu",
              name="dense_1_custom"),
        Dense(10, name="output_layer_custom")
    ], name="mnist_model_custom_tuned")

    hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log', default=LEARNING_RATE)
    # hp.Int("batch_size", min_value=32, max_value=128, step=32, default=BATCH_SIZE) # Definido no run_trial, mas pode ser declarado aqui para o search_space_summary
    return model

# Configuração do Tuner Customizado
tuner_custom = CustomLoopTuner(
    hypermodel=build_model_for_custom_loop,
    oracle=kt.oracles.HyperbandOracle( # CORREÇÃO APLICADA AQUI
        objective=kt.Objective('val_accuracy', direction='max'),
        max_epochs=NUM_EPOCHS,
        factor=3,
        seed=RANDOM_SEED,
    ),
    directory='custom_loop_tuner_results_final_v3', # Novo diretório para evitar conflitos
    project_name='mnist_custom_loop_hyperband_v3',
    overwrite=True
)

# Exibe um sumário do espaço de busca
print("\n### Iniciando Ajuste de Hiperparâmetros com Loop de Treinamento Personalizado ###")
tuner_custom.search_space_summary()

# Inicia a busca.
print("\nIniciando a busca com CustomLoopTuner...")
tuner_custom.search(train_dataset=train_dataset, val_dataset=val_dataset, epochs=NUM_EPOCHS)

Trial 30 Complete [00h 00m 13s]
val_accuracy: 0.9775000214576721

Best val_accuracy So Far: 0.984333336353302
Total elapsed time: 00h 11m 27s


# **5. Análise dos Resultados e Avaliação Final**

Após a conclusão da busca, `tuner_custom.results_summary()` exibe um resumo dos melhores trials encontrados.
* `tuner_custom.get_best_hyperparameters(num_trials=1)[0]` recupera o conjunto dos melhores hiperparâmetros.
* `tuner_custom.hypermodel.build(best_hps_custom)` constrói uma nova instância do modelo usando esses melhores hiperparâmetros.

Finalmente o melhor modelo é avaliado no conjunto de teste (`test_dataset`) usando uma função de avaliação customizada (`final_evaluation_step`), que é semelhante ao `val_step_custom` mas opera no conjunto de teste e usa métricas separadas. O `optimal_batch_size` encontrado pelo tuner é usado para _re-batchar_ o `test_dataset`.

In [8]:
# Obter e exibir os melhores hiperparâmetros
print("\nBusca com CustomLoopTuner concluída.")
tuner_custom.results_summary(num_trials=5)

best_hps_custom_list = tuner_custom.get_best_hyperparameters(num_trials=1)
if best_hps_custom_list:
    best_hps_custom = best_hps_custom_list[0]
    print(f"\nMelhores Hiperparâmetros encontrados (Custom Loop):")
    for hp_name, hp_value in best_hps_custom.values.items():
        print(f"  {hp_name}: {hp_value}")

    best_model_custom = tuner_custom.hypermodel.build(best_hps_custom)
    best_model_custom.summary()

    print("\nAvaliando o melhor modelo customizado no conjunto de teste...")
    test_loss_metric_final = Mean(name='test_loss_final')
    test_accuracy_metric_final = SparseCategoricalAccuracy(name='test_accuracy_final')

    loss_fn_eval = SparseCategoricalCrossentropy(from_logits=True)

    optimal_batch_size = best_hps_custom.get('batch_size')
    if optimal_batch_size is None:
        optimal_batch_size = BATCH_SIZE
        print(f"Alerta: 'batch_size' não encontrado nos HPs ótimos, usando default: {optimal_batch_size}")

    current_test_dataset = test_dataset.unbatch().batch(optimal_batch_size).prefetch(tf.data.AUTOTUNE)

    @tf.function
    def final_evaluation_step(x, y, model_to_eval, loss_metric, acc_metric):
        logits = model_to_eval(x, training=False)
        loss = loss_fn_eval(y, logits)
        if model_to_eval.losses:
            loss += tf.add_n(model_to_eval.losses)
        loss_metric.update_state(loss)
        acc_metric.update_state(y, logits)

    for x_batch_test, y_batch_test in current_test_dataset:
        final_evaluation_step(x_batch_test, y_batch_test, best_model_custom, test_loss_metric_final, test_accuracy_metric_final)

    print(f"Resultado Final no Teste (Custom Loop):")
    print(f"  Perda Teste: {test_loss_metric_final.result():.4f}")
    print(f"  Acurácia Teste: {test_accuracy_metric_final.result():.4f}")

else:
    print("Nenhum hiperparâmetro foi encontrado para o loop customizado.")


Busca com CustomLoopTuner concluída.
Results summary
Results in custom_loop_tuner_results_final_v3/mnist_custom_loop_hyperband_v3
Showing 5 best trials
Objective(name="val_accuracy", direction="max")

Trial 0026 summary
Hyperparameters:
units_custom: 256
learning_rate: 0.0005510469719519641
batch_size: 32
tuner/epochs: 10
tuner/initial_epoch: 0
tuner/bracket: 0
tuner/round: 0
Score: 0.984333336353302

Trial 0013 summary
Hyperparameters:
units_custom: 256
learning_rate: 0.0007767449713530701
batch_size: 64
tuner/epochs: 4
tuner/initial_epoch: 2
tuner/bracket: 2
tuner/round: 1
tuner/trial_id: 0007
Score: 0.9828333258628845

Trial 0002 summary
Hyperparameters:
units_custom: 128
learning_rate: 0.0012482904754698163
batch_size: 32
tuner/epochs: 2
tuner/initial_epoch: 0
tuner/bracket: 2
tuner/round: 0
Score: 0.9826666712760925

Trial 0019 summary
Hyperparameters:
units_custom: 128
learning_rate: 0.0010277342442932575
batch_size: 32
tuner/epochs: 4
tuner/initial_epoch: 0
tuner/bracket: 1
tun

Model: "mnist_model_custom_tuned"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_layer (Flatten)         │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1_custom (Dense)          │ (None, 256)            │       200,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer_custom (Dense)     │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 203,530 (795.04 KB)

 Trainable params: 203,530 (795.04 KB)

 Non-trainable params: 0 (0.00 B)


Avaliando o melhor modelo customizado no conjunto de teste...
Resultado Final no Teste (Custom Loop):
  Perda Teste: 2.3898
  Acurácia Teste: 0.1010


# **Referências**

* LI, L. et al. **Hyperband: Bandit-based configuration evaluation for hyperparameter optimization**. In: International Conference on Learning Representations (ICLR) , 2017. Disponível em: https://openreview.net/pdf?id=ry18Ww5ee Acesso em: 10 de Maio de 2025.

* LI, L. et al. **Hyperband: A Novel Bandit-Based Approach to Hyperparameter Optimization**. Journal of Machine Learning Research, v. 18, p. 1-52, 2018. Disponível em: https://www.jmlr.org/papers/volume18/16-558/16-558.pdf Acesso em: 10 de Maio de 2025.